# Семинар 2. 

## Иерархические модели

In [ ]:
import numpy as np
import pymc as pm
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import arviz as az
import pytensor
import pytensor.tensor as pt
import warnings
warnings.filterwarnings("ignore")
az.style.use("seaborn-v0_8-whitegrid")

$$
Y = A L^\beta
$$

In [ ]:
# визуализируем, как выглядит производительность в нашей модели 
np.random.seed(42)
β = 0.5 # параметр отдачи от труда; наиболее интересующий нас параметр
Lm = np.linspace(10, 40, 6) # допустимые значения труда
A = 15/Lm # технология 

L = np.exp(np.random.normal(np.log(Lm), 0.4, size=(20, len(Lm))))
Y = A * L ** β
Y += np.random.randn(*Y.shape) * 0.5
data = pd.DataFrame(dict(
    L=L.T.flatten(), Y=Y.T.flatten(), G=np.repeat(range(len(Lm)), Y.shape[0])
))

In [ ]:
sns.scatterplot(x="L", y="Y", data=data);

In [ ]:
sns.scatterplot(x="L", y="Y", hue="G", data=data, palette="Set1");

In [ ]:
sns.lmplot(x="L", y="Y", hue="G", data=data, palette="Set1");

In [ ]:
sns.lmplot(x="L", y="Y", hue="G", data=data, palette="Set1");
plt.loglog()

In [ ]:
sns.lmplot(x="L", y="Y", data=data);

## Модель с одной группой

### Априорные распределения

In [ ]:
sns.kdeplot(pm.Uniform.dist(size=1000).eval(), clip=[0, 1])

In [ ]:
sns.kdeplot(pm.Beta.dist(1, 1, size=1000).eval(), clip=[0, 1])

In [ ]:
sns.kdeplot(pm.Kumaraswamy.dist(10, 10, size=1000).eval(), clip=[0, 1])

In [ ]:
sns.kdeplot(pm.LogitNormal.dist(size=1000).eval(), clip=[0, 1])

Подбираем параметры априорного распределения. Делаем это на основе параметра `mass`, относительно которого ожидаем получить вероятность для заданных параметров

In [ ]:
opt_params = pm.find_constrained_prior( # итеративный поиск параметров, который подбирает модель под данные
    pm.Gamma, lower=0.1, upper=0.4, mass=0.75, init_guess={"alpha": 1, "beta": 10}
)

In [ ]:
opt_params

In [ ]:
opt_params = pm.find_constrained_prior(
    pm.LogNormal, lower=0.05, upper=0.15, mass=0.95, init_guess={"mu": 0, "sigma": 1}
    # fixed_params позволяет зафиксировать некоторые параметры, чтобы не искать их в процессе оптимизации; пример - nu для t-распределения (см. документацию)
)

In [ ]:
opt_params

In [ ]:
# можем вытащить один из параметров, например, mu, и посмотреть на его значение в исходной шкале
np.exp(opt_params["mu"])

In [ ]:
# строим модель с подобранными параметром mu 
with pm.Model() as model:
    sigma = pm.LogNormal("σ", **opt_params)

In [ ]:
# итоговый вид модели
model

$$
\begin{align*}
    \log Y_{g=0} &\sim \operatorname{Normal}(\log A_{g=0} + \log L \cdot \beta, \varepsilon)\\
    \varepsilon &\sim \operatorname{LogNormal}(-2, 1)\\
    \beta &\sim \operatorname{LogitNormal}(0, 1)\\
    A_{g=0} &\sim \operatorname{LogNormal}(0, 1)\\
\end{align*}
$$

In [ ]:
# убираем разбиение на группы, чтобы посмотреть на общую зависимость
data0 = data.query("G==0") 

In [ ]:
data0

In [ ]:
with pm.Model() as model1:
    a = pm.LogNormal("A", 0, 1)
    b = pm.LogitNormal("beta", 0, 1)
    e = pm.LogNormal("e", -2, 1)
    y = pm.LogNormal("obs", pt.log(a) + b*pt.log(data0.L), e, observed=data0.Y) # здесь добавил умножение на beta, иначе модели не сходились

In [ ]:
with model1:
    prior = pm.sample_prior_predictive()

In [ ]:
az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
with model1:
    trace = pm.sample()

In [ ]:
az.plot_trace_dist(trace);

### Фиксируем априорные распределения

In [ ]:
with pm.Model() as model1:
    a = pm.LogNormal("A", **pm.find_constrained_prior(pm.LogNormal, lower=0.5, upper=20, mass=0.95, init_guess={"mu": 0, "sigma": 1}))
    b = pm.LogitNormal("beta", 0, 1)
    e = pm.LogNormal("e", -2, 0.1)
    mu = pm.Deterministic("mu", pt.log(a) + b*np.log(data0.L))
    y = pm.LogNormal("obs", mu, e, observed=data0.Y)
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
plt.scatter(data0.L, prior.prior_predictive["obs"].sel(chain=0, draw=5)); # предсказание Y в зависимости от L + шум 
plt.scatter(data0.L, np.exp(prior.prior["mu"].sel(chain=0, draw=5)), color="red"); # предсказание среднего значения Y в зависимости от L

In [ ]:
# pm.do позволяет проверить распределение при фиксированных значениях параметров, например, e и beta
with pm.do(model1, {"e": 0.1, "beta": 0.01}): # можно убрать шум (e = 0.0), и формально делать предсказания
    prior1 = pm.sample_prior_predictive(1) # NB: pm.do возвращает новый контекст модели, с зафиксированными параметрамии

In [ ]:
# более простая модель, без шума и с зафиксированным beta
pm.do(model1, {"e": 0.1, "beta": 0.01})

In [ ]:
plt.scatter(data0.L, prior1.prior_predictive["obs"].sel(chain=0, draw=0));
plt.scatter(data0.L, np.exp(prior1.prior["mu"].sel(chain=0, draw=0)), color="red");

In [ ]:
with model1: # NB: ниже target_accept=0.8 обычно не делают, так как модели просто перестанут сходиться
    trace = pm.sample(target_accept=0.9) # target_accept позволяет регулировать шаг интегратора; можно считать как обратную к learning rate из классического ML
    az.plot_trace_dist(trace, var_names=['beta', 'A', 'e']) # mu нам не нужно, так как транслирует промежуточные вычисления; его в trace_plot не транслируем

## Воронка Нила

$$
\begin{align}
s &\sim \operatorname{HalfCauchy}(1)\\
m &\sim \operatorname{Normal}(0, s)
\end{align}
$$

In [ ]:
with pm.Model() as neals_funnel:
    s = pm.Exponential("s", 1)
    m = pm.Normal("m", 0, s)
    trace = pm.sample() # тут убрал target_accept=0.99, чтобы модель сошлась быстрее, но появились дивергенции, которые можно увидеть на графике ниже

In [ ]:
az.ess(trace)

In [ ]:
az.plot_pair(trace, 
            marginal=False,
            visuals={"divergence": True},
            #  aes_by_visuals={"divergence": {"color": "red"}}
            );

In [ ]:
with pm.Model() as no_neals_funnel:
    s = pm.Exponential("s", 1)
    m = pm.Deterministic("m", pm.Normal("mhat") * s)
    trace1 = pm.sample()

In [ ]:
az.ess(trace1)

In [ ]:
ax = az.plot_pair(trace, var_names=["m", "s"], triangle="lower", visuals= {"divergence": True});

In [ ]:
# Накладываем две воронки друг на друга
def get_flat(idata, var, group="posterior"):
    return getattr(idata, group).ds[var].values.flatten()

m1, s1 = get_flat(trace, "m"), get_flat(trace, "s")
m2, s2 = get_flat(trace1, "m"), get_flat(trace1, "s")

div1 = trace.sample_stats.ds["diverging"].values.flatten().astype(bool)
div2 = trace1.sample_stats.ds["diverging"].values.flatten().astype(bool)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(m1, s1, color="C1", alpha=0.6, s=15, label="trace")
ax.scatter(m2, s2, color="C0", alpha=0.6, s=15, label="trace1")
ax.scatter(m1[div1], s1[div1], facecolors="none", edgecolors="black", s=40)
ax.scatter(m2[div2], s2[div2], facecolors="none", edgecolors="black", s=40)
ax.set_xlabel("m")
ax.set_ylabel("s")
ax.legend()
plt.tight_layout()
plt.show()

$$
\begin{align}
s &\sim \operatorname{HalfCauchy}(1)\\
\bar m &\sim \operatorname{Normal}(0, 1)\\
m &= s \bar m
\end{align}
$$

## Иерархическая модель

In [ ]:
idx, index = pd.factorize(data.G)
idx

In [ ]:
index

In [ ]:
g_index, g_coord = pd.factorize(data.G)

In [ ]:
g_index

In [ ]:
fff = np.random.randn(6)

In [ ]:
fff[g_index]

In [ ]:
np.repeat(fff, [10, 2, 5, 2, 5, 9])

In [ ]:
# Здесь нужно сделать сначала модель с 1 группой, чтобы потом показать усложнение модели
# Надо оставить модель снизу как подсказку 

In [ ]:
with pm.Model(
    coords=dict(group=index, idx=data.index)
) as model2:
    a_pop = pm.LogNormal("Apop", **pm.find_constrained_prior(pm.LogNormal, lower=0.5, upper=20, mass=0.95, init_guess={"mu": 0, "sigma": 1}))
    # a_sigma = 0.1 #  pm.Exponential("a_sigma", 10)
    a_sigma = pm.Exponential("a_sigma", 10)
    a = pm.LogNormal("A", pt.log(a_pop), a_sigma, dims="group")
    b = pm.LogitNormal("beta", 0, 1)
    e = pm.LogNormal("e", -2, 0.1)
    mu = pm.Deterministic("mu", pt.log(a[g_index]) + b*np.log(data.L), dims="idx") 
    y = pm.LogNormal("obs", mu, e, observed=data.Y, dims="idx")
    prior = pm.sample_prior_predictive()
    az.plot_ppc_dist(prior, group="prior_predictive", visuals={"observed_dist": True});

In [ ]:
# визуализиурем, по A будет несколько групп
with model2:
    trace2 = pm.sample()
    az.plot_trace_dist(trace2, var_names=["beta", "Apop", "A"])

## Дискретные распределения

In [ ]:
# Смотрим, как строить смешенные модели
?pm.Mixture

In [ ]:
pm.Poisson.dist(mu=1)

In [ ]:
with pm.Model(coords=dict(idx=range(10))) as model:
    p = pm.LogitNormal("p", dims="idx")
    # p = p_group[group_index]
    w = pt.stack([p, 1-p], axis=1)

    # Так как нам нужен только logp, а не добавление RV в модель, нужно вызвать `.dist()`
    # Эти две формы эквивалентны, но вторая выигрывает за счёт векторизации
    components = [
        pm.Normal.dist(),
        pm.DiracDelta.dist(0.),
    ]
    # # `shape=(2,)` indicates 2 mixture components
    # components = pm.Poisson.dist(mu=pm.math.stack([lam1, lam2]), shape=(2,))
    choice = pm.Bernoulli("choice", p=p)
    like = pm.Mixture('like', w=w, comp_dists=components)
    prior = pm.sample_prior_predictive()

In [ ]:
p.type

In [ ]:
?pm.Potential

## Непрерывные распределения

In [ ]:
?pm.BetaBinomial